In [ ]:
!pip install -q -U langchain langchain-text-splitters tqdm gitpython

In [ ]:
import os
import shutil
import glob
from tqdm.notebook import tqdm
from git import Repo
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
from langchain_core.documents import Document
import pickle

# === КОНФИГУРАЦИЯ ===

# Куда клонируем временно
TEMP_DIR = "./hf_docs_temp"
# Куда сохраняем обработанные данные
OUTPUT_FILE = "hf_knowledge_base.pkl"

# Список библиотек и путей к документации внутри репо
# Формат: "Название": ("URL репозитория", "Путь к докам внутри")
LIBS_CONFIG = {
    "transformers": ("https://github.com/huggingface/transformers.git", "docs/source/en"),
    "peft": ("https://github.com/huggingface/peft.git", "docs/source"),
    "accelerate": ("https://github.com/huggingface/accelerate.git", "docs/source"),
    "trl": ("https://github.com/huggingface/trl.git", "docs/source"),
    "datasets": ("https://github.com/huggingface/datasets.git", "docs/source"),
}

# Параметры чанкинга
CHUNK_SIZE = 1000  # Символов
CHUNK_OVERLAP = 200 # Перекрытие для сохранения контекста

# === ФУНКЦИИ ===

def download_repo(lib_name, repo_url):
    """Клонирует репозиторий (shallow clone) во временную папку."""
    lib_path = os.path.join(TEMP_DIR, lib_name)

    if os.path.exists(lib_path):
        print(f"🔄 Папка {lib_name} уже существует, пропускаем скачивание.")
        return lib_path

    print(f"📥 Клонирование {lib_name}...")
    # depth=1 скачивает только последний коммит (экономит время и место)
    Repo.clone_from(repo_url, lib_path, depth=1)
    return lib_path

def load_and_process_docs():
    """Основной пайплайн обработки."""
    all_documents = []

    # 1. Создаем сплиттер, заточенный под Markdown
    splitter = RecursiveCharacterTextSplitter.from_language(
        language=Language.MARKDOWN,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )

    # Создаем папку если нет
    if not os.path.exists(TEMP_DIR):
        os.makedirs(TEMP_DIR)

    for lib_name, (repo_url, doc_path) in LIBS_CONFIG.items():
        print(f"\n--- Обработка {lib_name} ---")

        # 2. Скачивание
        local_path = download_repo(lib_name, repo_url)
        full_doc_path = os.path.join(local_path, doc_path)

        # 3. Поиск всех .md файлов
        md_files = glob.glob(os.path.join(full_doc_path, "**/*.md"), recursive=True)
        print(f"📄 Найдено {len(md_files)} markdown файлов.")

        lib_docs = []
        for file_path in tqdm(md_files, desc=f"Чтение {lib_name}"):
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read()

                # Пропускаем пустые файлы или индексные файлы (обычно index.md - это просто оглавление)
                if len(content) < 50:
                    continue

                # Создаем сырой документ
                # Важно: добавляем source (путь) и lib_name (для роутера)
                rel_path = os.path.relpath(file_path, full_doc_path)
                doc = Document(
                    page_content=content,
                    metadata={
                        "source": rel_path,      # Например: "quicktour.md"
                        "lib_name": lib_name,    # Ключевое поле: "transformers"
                        "full_path": file_path
                    }
                )
                lib_docs.append(doc)
            except Exception as e:
                print(f"⚠️ Ошибка чтения {file_path}: {e}")

        # 4. Чанкинг документов библиотеки
        chunks = splitter.split_documents(lib_docs)
        print(f"✂️  Создано {len(chunks)} чанков для {lib_name}")
        all_documents.extend(chunks)

    return all_documents

In [ ]:
print("🚀 Старт Sprint 1: Data Pipeline")

docs = load_and_process_docs()

print(f"\n✅ Обработка завершена! Всего чанков: {len(docs)}")

# Сохраняем в файл для следующего этапа
with open(OUTPUT_FILE, "wb") as f:
    pickle.dump(docs, f)

print(f"💾 База знаний сохранена в {OUTPUT_FILE}")

# Пример того, как выглядит чанк
print("\n🧐 Пример случайного чанка:")
sample = docs[len(docs)//2]
print(f"Source Lib: {sample.metadata['lib_name']}")
print(f"File: {sample.metadata['source']}")
print("-" * 20)
print(sample.page_content[:200] + "...")

# Очистка (по желанию)
# shutil.rmtree(TEMP_DIR)

🚀 Старт Sprint 1: Data Pipeline

--- Обработка transformers ---
📥 Клонирование transformers...
📄 Найдено 619 markdown файлов.


Чтение transformers:   0%|          | 0/619 [00:00<?, ?it/s]

✂️  Создано 7253 чанков для transformers

--- Обработка peft ---
📥 Клонирование peft...
📄 Найдено 67 markdown файлов.


Чтение peft:   0%|          | 0/67 [00:00<?, ?it/s]

✂️  Создано 714 чанков для peft

--- Обработка accelerate ---
📥 Клонирование accelerate...
📄 Найдено 58 markdown файлов.


...

✂️  Создано 712 чанков для accelerate

--- Обработка trl ---
📥 Клонирование trl...
📄 Найдено 59 markdown файлов.


Чтение trl:   0%|          | 0/59 [00:00<?, ?it/s]

✂️  Создано 965 чанков для trl

--- Обработка datasets ---
📥 Клонирование datasets...
📄 Найдено 4 markdown файлов.


Чтение datasets:   0%|          | 0/4 [00:00<?, ?it/s]

✂️  Создано 16 чанков для datasets

✅ Обработка завершена! Всего чанков: 9660
💾 База знаний сохранена в hf_knowledge_base.pkl

🧐 Пример случайного чанка:
Source Lib: transformers
File: model_doc/glmasr.md
--------------------
# loading audio directly from dataset
ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
ds = ds.cast_column("audio", Audio(sampling_rate=processor.feature_ext...


In [ ]:
!pip install -q -U langchain-chroma langchain-community sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 k

In [ ]:
import pickle
import os
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# === КОНФИГУРАЦИЯ ===
INPUT_FILE = "hf_knowledge_base.pkl"
CHROMA_PATH = "hf_vector_db" # Папка, где будет лежать база
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

def run_sprint_2():
    # 1. Загружаем чанки из Спринта 1
    print(f"📂 Загрузка чанков из {INPUT_FILE}...")
    with open(INPUT_FILE, "rb") as f:
        docs = pickle.load(f)
    print(f"✅ Загружено {len(docs)} документов.")

    # 2. Инициализируем эмбеддинги
    print(f"🧠 Инициализация модели эмбеддингов: {EMBEDDING_MODEL}...")
    # Если есть GPU, используем его (device="cuda")
    encode_kwargs = {'normalize_embeddings': True} # Для лучшего косинусного сходства
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={'device': 'cpu'}, # В Colab можно сменить на 'cuda'
        encode_kwargs=encode_kwargs
    )

    # 3. Создаем векторную базу
    # Если папка уже есть, мы ее просто подгрузим, а не будем создавать заново
    if os.path.exists(CHROMA_PATH):
        print(f"📦 Индекс найден в {CHROMA_PATH}, загружаем...")
        vectorstore = Chroma(
            persist_directory=CHROMA_PATH,
            embedding_function=embeddings
        )
    else:
        print(f"🔨 Создание нового индекса (это может занять 2-5 минут для 10k чанков)...")
        vectorstore = Chroma.from_documents(
            documents=docs,
            embedding=embeddings,
            persist_directory=CHROMA_PATH
        )
        print(f"✅ База данных создана и сохранена в {CHROMA_PATH}")

    return vectorstore

# === ТЕСТИРОВАНИЕ ПОИСКА ===

def test_retrieval(vectorstore, query):
    print(f"\n🔍 Вопрос: {query}")
    print("-" * 30)

    # Поиск топ-3 похожих документов
    results = vectorstore.similarity_search(query, k=3)

    for i, doc in enumerate(results):
        lib = doc.metadata.get('lib_name', 'unknown')
        source = doc.metadata.get('source', 'unknown')
        print(f"📌 Результат #{i+1} [Библиотека: {lib.upper()}]")
        print(f"📄 Файл: {source}")
        print(f"📝 Текст: {doc.page_content[:200]}...")
        print("-" * 10)

# === ЗАПУСК ===

if __name__ == "__main__":
    v_db = run_sprint_2()

    # Тестируем на трех разных библиотеках
    test_retrieval(v_db, "How to use LoRA with PeftModel?")
    test_retrieval(v_db, "How to load a dataset from local disk?")
    test_retrieval(v_db, "What is Mixed Precision training in Accelerate?")

📂 Загрузка чанков из hf_knowledge_base.pkl...
✅ Загружено 9660 документов.
🧠 Инициализация модели эмбеддингов: sentence-transformers/all-MiniLM-L6-v2...


/tmp/ipython-input-2031886161.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔨 Создание нового индекса (это может занять 2-5 минут для 10k чанков)...
✅ База данных создана и сохранена в hf_vector_db

🔍 Вопрос: How to use LoRA with PeftModel?
------------------------------
📌 Результат #1 [Библиотека: PEFT]
📄 Файл: developer_guides/lora.md
📝 Текст: # next, load the LoRA adapter for German
peft_model.load_adapter(<path>, adapter_name="adapter_de")...
----------
📌 Результат #2 [Библиотека: PEFT]
📄 Файл: developer_guides/lora.md
📝 Текст: # LoRA

LoRA is low-rank decomposition method to reduce the number of trainable parameters which speeds up finetuning large models and uses less memory. In PEFT, using LoRA is as easy as setting up a ...
----------
📌 Результат #3 [Библиотека: PEFT]
📄 Файл: developer_guides/custom_models.md
📝 Текст: ```

> [!TIP]
> When you call [`get_peft_model`], you will see a warning because PEFT does not recognize the targeted module type. In this case, you can ignore this warning.

By supplying a custom map...
----------

🔍 Вопрос: How to load 

In [ ]:
!pip install -q -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 3.9 MB/s eta 0:00:00


In [ ]:
import os
import pickle
from typing import List
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from pydantic import BaseModel, Field

# === 1. ИНИЦИАЛИЗАЦИЯ (Заполни ключи!) ===
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "RAG-Hugging-face-docs"
os.environ["LANGCHAIN_API_KEY"] = "secret"
OPENROUTER_API_KEY = "sk-or-v1-secret"

llm = ChatOpenAI(
    model="z-ai/glm-4.5-air:free",
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
)

# Подгружаем базу из Спринта 2
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory="hf_vector_db", embedding_function=embeddings)

# === 2. ROUTER (Определяем библиотеку) ===
# Упрощаем промпт: просим только название библиотеки одним словом
router_prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты эксперт Hugging Face. Определи, к какой библиотеке относится вопрос: transformers, peft, accelerate, trl, datasets. Если вопрос общий, ответь 'general'. Пиши ТОЛЬКО ОДНО СЛОВО — название библиотеки."),
    ("user", "{question}")
])

# Используем обычный StrOutputParser вместо structured_output
router_chain = router_prompt | llm | StrOutputParser()

# === 3. RAG FUSION (Генерация вариаций) ===
fusion_prompt = ChatPromptTemplate.from_messages([
    ("system", "Сгенерируй 3 разных варианта одного и того же вопроса, чтобы найти максимум информации в документации. Пиши только вопросы, по одному на строку."),
    ("user", "{question}")
])

fusion_chain = fusion_prompt | llm | StrOutputParser() | (lambda x: x.split("\n"))

# === 4. УМНЫЙ ПОИСК (Filtered Retrieval) ===
def smart_retriever(input_data):
    question = input_data["question"]

    # 1. Роутинг (получаем строку, чистим от пробелов и кавычек)
    lib_response = router_chain.invoke({"question": question})
    lib = lib_response.strip().lower().replace("'", "").replace('"', "")

    # Проверка на допустимые значения (защита от галлюцинаций)
    allowed_libs = ["transformers", "peft", "accelerate", "trl", "datasets"]
    if lib not in allowed_libs:
        lib = "general"

    # 2. Генерация вариаций для Fusion
    queries = fusion_chain.invoke({"question": question})
    # Добавляем проверку, если fusion_chain вернул список или строку
    if isinstance(queries, str):
        queries = queries.split("\n")
    queries.append(question)

    # 3. Поиск с фильтром по метаданным
    search_kwargs = {"k": 3}
    if lib != "general":
        search_kwargs["filter"] = {"lib_name": lib}

    all_docs = []
    for q in queries:
        if q.strip(): # Пропускаем пустые строки
            all_docs.extend(vectorstore.similarity_search(q, **search_kwargs))

    unique_docs = {doc.page_content: doc for doc in all_docs}.values()
    return {"context": list(unique_docs), "question": question, "lib": lib}

# === 5. ФИНАЛЬНЫЙ ОТВЕТ ===
rag_prompt = ChatPromptTemplate.from_template("""
Ты — технический ассистент Hugging Face. Используй предоставленный контекст (из библиотеки {lib}), чтобы ответить на вопрос.
Если в контексте нет ответа, скажи, что ты не знаешь.
Пиши четко, с примерами кода, если они есть в контексте.

Контекст:
{context}

Вопрос: {question}
Ответ:
""")

final_chain = (
    {"question": RunnablePassthrough()}
    | RunnableLambda(smart_retriever)
    | rag_prompt
    | llm
    | StrOutputParser()
)

# === ТЕСТ ===
if __name__ == "__main__":
    test_q = "How to apply LoRA on a Qwen model using PEFT?"
    print(f"🚀 Запуск Advanced RAG...")
    response = final_chain.invoke(test_q)
    print("\n" + "="*50)
    print(response)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 Запуск Advanced RAG...



Based on the provided context, here's how to apply LoRA to a **Qwen model** using PEFT, with specific guidance and code examples:

### Key Steps:
1. **Identify Target Modules**  
   Qwen models share similar architecture to other transformer models. According to the context, you should determine the correct `target_modules` by checking PEFT's constants or the model's architecture. For Qwen, common targets are typically `["q_proj", "v_proj"]` (similar to Mistral models).

2. **Set Up `LoraConfig`**  
   Configure LoRA parameters targeting the appropriate modules.

3. **Apply LoRA with `get_peft_model`**  
   Wrap your base model with the LoRA configuration.

---

### Example Code:
```python
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

# 1. Load the Qwen base model
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B")

# 2. Define LoRA config (targeting Qwen's common modules)
config = LoraConfig(
    target_modules=["q_proj", "v_proj"